# 06 — Modularity Analysis
**Purpose:** Identify natural feature clusters for modular builds. Compare
structural, behavioral, and organisational module boundaries.

**Inputs:** `target_metrics.parquet`, `edge_list.parquet`, `git_commit_log.parquet` (optional),
            `contributor_target_commits.parquet`, `target_ownership.parquet` (optional)

**Outputs:** `data/intermediate/communities.parquet`,
             `data/intermediate/feature_configurations.parquet`,
             `data/intermediate/gephi/cochange_graph.gexf`

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram

from buildanalysis.graphs import build_dependency_graph
from buildanalysis.modularity import (
    build_feature_configurations,
    compare_community_methods,
    compute_conway_alignment,
    compute_modularity_score,
    cut_dendrogram,
    detect_communities_louvain,
    detect_communities_spectral,
    hierarchical_clustering,
)
from buildanalysis.git import (
    compute_cochange_matrix,
    compute_file_to_target_map,
    infer_team_assignments,
)
from buildanalysis.export import export_cochange_graph, export_dependency_graph
from buildanalysis.loading import BuildDataset

warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 100})

DATA_DIR = Path("../../data/processed")
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)
tm = ds.target_metrics
el = ds.edge_list

HAS_GIT = ds.has_file("git_commit_log")

print(f"target_metrics: {tm.shape[0]:,} targets")
print(f"edge_list:      {el.shape[0]:,} edges")
print(f"Git commit log: {'available' if HAS_GIT else 'not found'}")

# Check optional intermediate files
for f in ["contributor_groups", "target_ownership"]:
    p = INTERMEDIATE_DIR / f"{f}.parquet"
    print(f"{f}: {'exists' if p.exists() else 'MISSING'}")

In [ ]:
bg = build_dependency_graph(tm, el, direct_only=True)
print(f"Graph: {bg.n_targets} targets, {bg.n_edges} direct edges")

## 1. Structural Community Detection

Run Louvain at multiple resolutions, spectral clustering with eigengap heuristic,
and hierarchical clustering. Compare methods and select the best partition.

In [ ]:
# Louvain at multiple resolutions
louvain_results = {}
for res in [0.5, 1.0, 1.5, 2.0]:
    communities = detect_communities_louvain(bg, resolution=res)
    louvain_results[f"louvain_r{res}"] = communities
    n_comm = communities["community"].nunique()
    sizes = communities.groupby("community").size()
    print(f"Louvain r={res}: {n_comm} communities, sizes [{sizes.min()}, {sizes.median():.0f}, {sizes.max()}]")

In [ ]:
# Spectral clustering with eigengap heuristic
spectral_communities = detect_communities_spectral(bg)
n_spectral = spectral_communities["community"].nunique()
sizes = spectral_communities.groupby("community").size()
print(f"Spectral (eigengap k={n_spectral}): sizes [{sizes.min()}, {sizes.median():.0f}, {sizes.max()}]")

In [ ]:
# Hierarchical clustering + dendrogram
Z, hc_nodes = hierarchical_clustering(bg)

fig, ax = plt.subplots(figsize=(14, 6))
p = min(30, len(hc_nodes) - 1)
dendrogram(Z, truncate_mode="lastp", p=p, leaf_rotation=90, leaf_font_size=8, ax=ax)
ax.set_title(f"Hierarchical Clustering Dendrogram (top {p} merges)")
ax.set_ylabel("Jaccard distance")
fig.tight_layout()
plt.show()

# Cut at several k values for comparison
hc_results = {}
for k in [5, 10, 15, 20]:
    if k < len(hc_nodes):
        hc_df = cut_dendrogram(Z, hc_nodes, n_clusters=k)
        hc_results[f"hierarchical_k{k}"] = hc_df
        print(f"Hierarchical k={k}: {hc_df['community'].nunique()} communities")

In [ ]:
# Compare all methods
all_methods = {**louvain_results, "spectral": spectral_communities, **hc_results}
comparison_df = compare_community_methods(bg, all_methods)
print("COMMUNITY DETECTION METHOD COMPARISON")
print("=" * 100)
print(comparison_df.to_string(index=False))

In [ ]:
# Select the best method: highest modularity among methods with practical community sizes (5-20 communities)
n_targets = bg.n_targets
practical = comparison_df[
    (comparison_df["n_communities"] >= 5)
    & (comparison_df["n_communities"] <= 20)
    & (comparison_df["min_size"] >= 1)
].copy()

if len(practical) > 0:
    best_row = practical.sort_values("modularity", ascending=False).iloc[0]
else:
    # Fall back to highest modularity overall
    best_row = comparison_df.iloc[0]

best_method = best_row["method"]
best_communities_df = all_methods[best_method]
best_score = compute_modularity_score(bg, best_communities_df)

print(f"SELECTED METHOD: {best_method}")
print(f"  Communities:            {best_score['n_communities']}")
print(f"  Modularity:             {best_score['graph_modularity']:.4f}")
print(f"  Inter-community edges:  {best_score['inter_community_edge_fraction']:.3f}")
print(f"  Avg self-containment:   {best_score['avg_self_containment']:.3f}")
print(f"  Size range:             [{best_score['min_community_size']}, {best_score['median_community_size']:.0f}, {best_score['max_community_size']}]")

## 2. Characterise Structural Communities

For each community: target count, dominant target types, total build time,
dominant team, top directories, and self-containment ratio.

In [ ]:
# Load team assignments
try:
    team_df = ds.load_intermediate("target_ownership")
    team_col = "primary_team" if "primary_team" in team_df.columns else "top_contributor"
    print(f"Loaded target_ownership (using column '{team_col}')")
except FileNotFoundError:
    if HAS_GIT:
        file_to_target = compute_file_to_target_map(ds.file_metrics)
        team_df = infer_team_assignments(ds.git_commit_log, file_to_target)
        team_col = "primary_team"
        print("Inferred team assignments from git log")
    else:
        team_df = pd.DataFrame({"cmake_target": tm["cmake_target"], "primary_team": "unknown"})
        team_col = "primary_team"
        print("No git data or target_ownership — all targets assigned to 'unknown'")

team_map = team_df.set_index("cmake_target")[team_col].to_dict()

In [ ]:
# Build per-community summary
comm_with_meta = best_communities_df.merge(
    tm[["cmake_target", "target_type", "total_build_time_ms", "file_count"]],
    on="cmake_target",
    how="left",
)
comm_with_meta["team"] = comm_with_meta["cmake_target"].map(team_map).fillna("unknown")

# Extract top-level directory from target name as a proxy for source directory
comm_with_meta["directory"] = comm_with_meta["cmake_target"].apply(
    lambda t: t.rsplit("::", 1)[0] if "::" in t else t.split("_")[0] if "_" in t else t
)

# Compute self-containment per community
undirected = bg.graph.to_undirected()
comm_lookup = best_communities_df.set_index("cmake_target")["community"].to_dict()

community_rows = []
for cid, group in comm_with_meta.groupby("community"):
    comm_nodes = set(group["cmake_target"])
    internal_edges = 0
    total_edges = 0
    for node in comm_nodes:
        if node in undirected:
            for neighbor in undirected.neighbors(node):
                total_edges += 1
                if neighbor in comm_nodes:
                    internal_edges += 1
    sc = internal_edges / total_edges if total_edges > 0 else 1.0

    dominant_type = group["target_type"].mode().iloc[0] if len(group) > 0 else "unknown"
    dominant_team = group["team"].mode().iloc[0] if len(group) > 0 else "unknown"
    dominant_dir = group["directory"].mode().iloc[0] if len(group) > 0 else "unknown"

    community_rows.append({
        "community_id": cid,
        "size": len(group),
        "build_time_ms": group["total_build_time_ms"].sum(),
        "dominant_type": dominant_type,
        "dominant_team": dominant_team,
        "dominant_directory": dominant_dir,
        "self_containment_ratio": sc,
    })

community_summary = pd.DataFrame(community_rows).sort_values("size", ascending=False)
community_summary.index = range(1, len(community_summary) + 1)

print("STRUCTURAL COMMUNITY SUMMARY")
print("=" * 110)
print(community_summary.to_string())

In [ ]:
# Community size distribution
fig, ax = plt.subplots(figsize=(10, 5))
sizes = community_summary.sort_values("community_id")
ax.bar(sizes["community_id"].astype(str), sizes["size"], color="steelblue", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Community")
ax.set_ylabel("Target count")
ax.set_title("Community Size Distribution")
fig.tight_layout()
plt.show()

## 3. Cross-Community Coupling

Identify edges crossing community boundaries, the most-coupled community pairs,
and "bridge targets" with many cross-community edges.

In [ ]:
# Cross-community edges
direct_edges = el[el["is_direct"]].copy()
direct_edges["src_community"] = direct_edges["source_target"].map(comm_lookup)
direct_edges["dst_community"] = direct_edges["dest_target"].map(comm_lookup)
cross_edges = direct_edges[direct_edges["src_community"] != direct_edges["dst_community"]].copy()

print(f"Total direct edges:        {len(direct_edges):,}")
print(f"Cross-community edges:     {len(cross_edges):,} ({len(cross_edges)/len(direct_edges)*100:.1f}%)")
print(f"Intra-community edges:     {len(direct_edges) - len(cross_edges):,}")

# Most-coupled community pairs
pair_counts = (
    cross_edges.groupby(["src_community", "dst_community"])
    .size()
    .reset_index(name="edge_count")
    .sort_values("edge_count", ascending=False)
)

print(f"\nTOP 15 MOST-COUPLED COMMUNITY PAIRS")
print("=" * 60)
print(pair_counts.head(15).to_string(index=False))

In [ ]:
# Community coupling heatmap
n_comms = best_communities_df["community"].nunique()
comm_ids = sorted(best_communities_df["community"].unique())

coupling_matrix = pd.DataFrame(0, index=comm_ids, columns=comm_ids, dtype=int)
for _, row in pair_counts.iterrows():
    coupling_matrix.at[row["src_community"], row["dst_community"]] = row["edge_count"]

fig, ax = plt.subplots(figsize=(max(8, n_comms * 0.6), max(6, n_comms * 0.5)))
sns.heatmap(
    coupling_matrix,
    annot=n_comms <= 20,
    fmt="d",
    cmap="YlOrRd",
    ax=ax,
    square=True,
    linewidths=0.5,
)
ax.set_title("Cross-Community Edge Count (source → dest)")
ax.set_xlabel("Destination community")
ax.set_ylabel("Source community")
fig.tight_layout()
plt.show()

In [ ]:
# Bridge targets — targets with the most cross-community edges
bridge_sources = cross_edges["source_target"].value_counts()
bridge_dests = cross_edges["dest_target"].value_counts()
bridge_total = (bridge_sources.add(bridge_dests, fill_value=0)).sort_values(ascending=False)

bridge_df = pd.DataFrame({
    "cmake_target": bridge_total.index,
    "cross_community_edges": bridge_total.values.astype(int),
})
bridge_df["community"] = bridge_df["cmake_target"].map(comm_lookup)
bridge_df["target_type"] = bridge_df["cmake_target"].map(
    tm.set_index("cmake_target")["target_type"].to_dict()
)

print("TOP 20 BRIDGE TARGETS (most cross-community edges)")
print("=" * 90)
print(bridge_df.head(20).to_string(index=False))

# PUBLIC cross-community edges
if "cmake_visibility" in direct_edges.columns:
    public_cross = cross_edges[cross_edges["cmake_visibility"] == "PUBLIC"]
    print(f"\nPUBLIC cross-community edges: {len(public_cross):,} of {len(cross_edges):,}")
    print("  (candidates for narrowing to PRIVATE to reduce coupling)")

## 4. Behavioral Community Detection (Co-change)

Compute target-level co-change coupling from git history, then detect communities
on the co-change graph. Requires `git_commit_log.parquet`.

In [ ]:
import networkx as nx

cochange_df = None
behavioral_communities = None

if HAS_GIT:
    file_to_target = compute_file_to_target_map(ds.file_metrics)
    git_log = ds.git_commit_log

    cochange_df = compute_cochange_matrix(
        git_log, file_to_target=file_to_target, level="target", min_cochanges=3
    )
    print(f"Co-change pairs (all): {len(cochange_df):,}")

    # Filter to meaningful coupling (PMI > 1.0)
    cochange_strong = cochange_df[cochange_df["pmi"] > 1.0]
    print(f"Co-change pairs (PMI > 1.0): {len(cochange_strong):,}")

    if len(cochange_strong) > 0:
        # Build co-change graph
        cochange_graph = nx.Graph()
        for _, row in cochange_strong.iterrows():
            cochange_graph.add_edge(row["item_a"], row["item_b"], weight=row["pmi"])

        print(f"Co-change graph: {cochange_graph.number_of_nodes()} nodes, {cochange_graph.number_of_edges()} edges")

        # Run Louvain on the co-change graph
        behav_comms = nx.community.louvain_communities(cochange_graph, resolution=1.0, seed=42)
        behav_rows = []
        for idx, comm in enumerate(behav_comms):
            for target in comm:
                behav_rows.append({"cmake_target": target, "community": idx})
        behavioral_communities = pd.DataFrame(behav_rows)

        n_behav = behavioral_communities["community"].nunique()
        behav_sizes = behavioral_communities.groupby("community").size()
        print(f"\nBehavioral communities: {n_behav}")
        print(f"  Size range: [{behav_sizes.min()}, {behav_sizes.median():.0f}, {behav_sizes.max()}]")
    else:
        print("No strong co-change pairs found — skipping behavioral community detection.")
else:
    print("⚠ git_commit_log.parquet not available — skipping behavioral community detection.")

## 5. Structural vs Behavioral Comparison

Compare structural (dependency-based) and behavioral (co-change-based) communities.
Misaligned targets structurally belong to one module but behaviorally belong to another.

In [ ]:
if behavioral_communities is not None:
    alignment = compute_conway_alignment(best_communities_df, behavioral_communities)

    print("STRUCTURAL vs BEHAVIORAL ALIGNMENT")
    print("=" * 50)
    print(f"  Adjusted Rand Index (ARI): {alignment['adjusted_rand_index']:.4f}")
    print(f"  Normalised Mutual Info:    {alignment['normalized_mutual_info']:.4f}")
    print(f"  Targets compared:          {alignment['n_targets_compared']}")
    print()
    if alignment["adjusted_rand_index"] > 0.6:
        print("Strong alignment — structural modules match how code is changed together.")
    elif alignment["adjusted_rand_index"] > 0.3:
        print("Moderate alignment — some structural modules diverge from change patterns.")
    else:
        print("Weak alignment — structural boundaries do not reflect co-change behaviour.")

    # Find misaligned targets
    merged = best_communities_df.merge(
        behavioral_communities, on="cmake_target", suffixes=("_struct", "_behav")
    )
    # Group behavioral communities by their dominant structural community to align labels
    behav_to_struct = merged.groupby("community_behav")["community_struct"].agg(lambda x: x.mode().iloc[0])
    merged["expected_struct"] = merged["community_behav"].map(behav_to_struct)
    misaligned = merged[merged["community_struct"] != merged["expected_struct"]].copy()
    misaligned["team"] = misaligned["cmake_target"].map(team_map).fillna("unknown")

    print(f"\nMisaligned targets: {len(misaligned)} of {len(merged)} ({len(misaligned)/len(merged)*100:.1f}%)")

    if len(misaligned) > 0:
        misaligned_display = misaligned.sort_values("cmake_target").head(20)
        misaligned_display.index = range(1, len(misaligned_display) + 1)
        print(f"\nTOP 20 MISALIGNED TARGETS")
        print("=" * 100)
        print(misaligned_display[["cmake_target", "community_struct", "community_behav", "team"]].to_string())
else:
    print("⚠ Behavioral communities not available — skipping structural vs behavioral comparison.")

## 6. Conway's Law Alignment (Organisational)

Compare team-based groupings against structural communities. Identify which teams
are well-aligned with community structure and which are scattered across multiple
communities.

In [ ]:
# Build team-based "community" assignment for comparison
team_comm_df = team_df[["cmake_target", team_col]].copy()
team_comm_df = team_comm_df.rename(columns={team_col: "team"})

# Encode teams as integers for ARI/NMI
team_labels = team_comm_df["team"].astype("category")
team_comm_df["community"] = team_labels.cat.codes
team_label_map = dict(enumerate(team_labels.cat.categories))

# Compare structural communities vs team groupings
conway = compute_conway_alignment(best_communities_df, team_comm_df[["cmake_target", "community"]])

print("CONWAY'S LAW ALIGNMENT: Structural vs Organisational")
print("=" * 55)
print(f"  Adjusted Rand Index (ARI): {conway['adjusted_rand_index']:.4f}")
print(f"  Normalised Mutual Info:    {conway['normalized_mutual_info']:.4f}")
print(f"  Targets compared:          {conway['n_targets_compared']}")
print()
if conway["adjusted_rand_index"] > 0.6:
    print("Strong Conway alignment — teams own coherent structural modules.")
elif conway["adjusted_rand_index"] > 0.3:
    print("Moderate Conway alignment — some teams span multiple modules.")
else:
    print("Weak Conway alignment — team boundaries do not match structural modules.")

In [ ]:
# Per-team scatter: how many structural communities does each team span?
team_struct = best_communities_df.merge(team_comm_df[["cmake_target", "team"]], on="cmake_target", how="inner")

if team_struct.empty:
    print("⚠ No targets matched between communities and team assignments — check cmake_target naming.")
    print(f"  Communities sample: {best_communities_df['cmake_target'].head(3).tolist()}")
    print(f"  Team assignments sample: {team_comm_df['cmake_target'].head(3).tolist()}")
    team_scatter = pd.DataFrame(columns=["team", "n_targets", "n_communities_spanned", "dominant_community", "alignment_fraction"])
else:
    team_scatter_rows = []
    for team_name, group in team_struct.groupby("team"):
        comms = group["community"].unique()
        dominant_comm = group["community"].mode().iloc[0]
        aligned_frac = (group["community"] == dominant_comm).mean()
        team_scatter_rows.append({
            "team": team_name,
            "n_targets": len(group),
            "n_communities_spanned": len(comms),
            "dominant_community": dominant_comm,
            "alignment_fraction": aligned_frac,
        })

    team_scatter = pd.DataFrame(team_scatter_rows).sort_values("alignment_fraction", ascending=True)
    team_scatter.index = range(1, len(team_scatter) + 1)

    print("TEAM-COMMUNITY ALIGNMENT")
    print("=" * 90)
    print(team_scatter.to_string())

    print(f"\nWell-aligned teams (>80% in one community): "
          f"{(team_scatter['alignment_fraction'] > 0.8).sum()} of {len(team_scatter)}")
    print(f"Scattered teams (<50% in one community):    "
          f"{(team_scatter['alignment_fraction'] < 0.5).sum()} of {len(team_scatter)}")

## 7. Feature Configuration Generation

For each community (potential feature), compute the full build set including
transitive dependencies. Identify the common "core" and assess which features
are genuinely independent.

In [ ]:
feature_configs = build_feature_configurations(
    bg,
    best_communities_df,
    timing=tm,
    time_col="total_build_time_ms",
)

display_cols = [
    "feature_id", "own_targets", "transitive_deps", "total_build_set",
    "build_fraction", "estimated_build_time_ms", "estimated_build_fraction_time",
]
print("FEATURE CONFIGURATIONS")
print("=" * 110)
print(feature_configs[display_cols].to_string(index=False, float_format="%.3f"))

In [ ]:
# Identify the common "core" — shared dependencies that appear in every feature's build set
import json

all_shared = set()
for _, row in feature_configs.iterrows():
    shared = set(json.loads(row["shared_deps_list"]))
    all_shared |= shared

# Targets that every feature needs (intersection of all build sets)
all_build_sets = []
for _, row in feature_configs.iterrows():
    own = set(json.loads(row["own_target_list"]))
    # Reconstruct full build set: own + transitive deps from graph
    build_set = set(own)
    for t in own:
        if t in bg.graph:
            build_set |= nx.descendants(bg.graph, t)
    all_build_sets.append(build_set)

if all_build_sets:
    core_targets = all_build_sets[0].copy()
    for bs in all_build_sets[1:]:
        core_targets &= bs

    print(f"Common core targets (in every feature's build set): {len(core_targets)}")
    print(f"  ({len(core_targets)/bg.n_targets*100:.1f}% of all targets)")
    print(f"Shared dependencies (used by 2+ features): {len(all_shared)}")

    # Core build time
    core_time = tm[tm["cmake_target"].isin(core_targets)]["total_build_time_ms"].sum()
    total_time = tm["total_build_time_ms"].sum()
    print(f"Core build time: {core_time:,.0f} ms ({core_time/total_time*100:.1f}% of total)")

In [ ]:
# Build fraction bar chart per feature
fig, ax = plt.subplots(figsize=(10, 5))
fc_sorted = feature_configs.sort_values("build_fraction", ascending=True)
colors = ["#2ecc71" if f < 0.3 else "#e67e22" if f < 0.7 else "#e74c3c" for f in fc_sorted["build_fraction"]]
ax.barh(
    fc_sorted["feature_id"].astype(str),
    fc_sorted["build_fraction"],
    color=colors,
    edgecolor="white",
    linewidth=0.3,
)
ax.set_xlabel("Build fraction (of total codebase)")
ax.set_ylabel("Feature ID")
ax.set_title("Build Fraction per Feature")
ax.axvline(0.3, color="green", linestyle="--", alpha=0.5, label="Independent (<30%)")
ax.axvline(0.7, color="red", linestyle="--", alpha=0.5, label="Heavily coupled (>70%)")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

# Summary
independent = feature_configs[feature_configs["build_fraction"] < 0.3]
coupled = feature_configs[feature_configs["build_fraction"] > 0.7]
print(f"Genuinely independent features (build_fraction < 30%): {len(independent)}")
print(f"Heavily coupled features (build_fraction > 70%):       {len(coupled)}")

## 8. Gephi Export — Co-change Graph

Export the target-level co-change graph as GEXF for Gephi visualisation, with
structural community labels and git churn as node attributes.

In [ ]:
if cochange_df is not None and len(cochange_df) > 0:
    from buildanalysis.git import compute_file_churn

    # Compute target-level churn for node attributes
    file_churn = compute_file_churn(git_log)
    file_to_target_map = compute_file_to_target_map(ds.file_metrics)

    target_churn = (
        file_churn[file_churn["source_file"].isin(file_to_target_map.index)]
        .assign(cmake_target=lambda df: df["source_file"].map(file_to_target_map))
        .groupby("cmake_target")
        .agg(n_commits=("n_commits", "sum"), total_churn=("total_churn", "sum"))
        .reset_index()
    )

    path = export_cochange_graph(
        cochange=cochange_df,
        target_metrics=tm,
        git_churn=target_churn,
        structural_communities=best_communities_df,
        min_pmi=1.0,
        output_path=INTERMEDIATE_DIR / "gephi" / "cochange_graph.gexf",
    )
    print(f"\nGEXF file: {path}")
else:
    print("⚠ No co-change data available — skipping Gephi export.")

### Gephi Usage Guide — Co-change Graph

**Setup:**
1. Open `data/intermediate/gephi/cochange_graph.gexf` in Gephi
2. Run **ForceAtlas2** layout with LinLog mode, scaling ~2.0
3. Size nodes by `n_commits` (more commits = larger node)
4. Colour nodes by `structural_community`
5. Set edge thickness by `pmi` (stronger coupling = thicker edge)

**What to look for:**
- **Structural-behavioral mismatches:** Nodes coloured by structural community
  that cluster differently under ForceAtlas2 layout (driven by co-change weights)
  suggest the structural boundary doesn't match real change patterns
- **Strong cross-community coupling:** Thick edges between differently-coloured
  nodes indicate co-change coupling that crosses structural module boundaries
- **Gephi Louvain comparison:** Run Gephi's built-in Louvain modularity and
  compare the resulting partition against the `structural_community` colouring

In [ ]:
# Candidate nodes for Gephi investigation
print("CANDIDATE NODES FOR GEPHI INVESTIGATION")
print()

if cochange_df is not None and len(cochange_df) > 0:
    # Strongest cross-community coupling pairs
    cochange_with_comm = cochange_df.copy()
    cochange_with_comm["comm_a"] = cochange_with_comm["item_a"].map(comm_lookup)
    cochange_with_comm["comm_b"] = cochange_with_comm["item_b"].map(comm_lookup)
    cross_cochange = cochange_with_comm[
        (cochange_with_comm["comm_a"] != cochange_with_comm["comm_b"])
        & (cochange_with_comm["pmi"] > 1.0)
    ].sort_values("pmi", ascending=False)

    print("--- Strongest cross-community co-change pairs (PMI > 1.0) ---")
    for _, row in cross_cochange.head(15).iterrows():
        print(f"  {row['item_a']:<40s} <-> {row['item_b']:<40s}  "
              f"PMI={row['pmi']:.2f}  count={row['cochange_count']}")
    print()

if behavioral_communities is not None and len(misaligned) > 0:
    print("--- Misaligned targets (structural != behavioral community) ---")
    for _, row in misaligned.head(15).iterrows():
        print(f"  {row['cmake_target']:<50s}  struct={row['community_struct']}  "
              f"behav={row['community_behav']}  team={row.get('team', '?')}")
else:
    print("--- No misaligned targets to report ---")

## 9. Persist Outputs

In [ ]:
# Save communities
out = ds.save_intermediate("communities", best_communities_df)
print(f"Saved communities.parquet: {best_communities_df.shape[0]} rows")
print(f"  -> {out}")

# Save feature configurations
out = ds.save_intermediate("feature_configurations", feature_configs)
print(f"Saved feature_configurations.parquet: {feature_configs.shape[0]} rows")
print(f"  -> {out}")

# Re-export dependency graph GEXF with final community labels
from buildanalysis.graphs import compute_centrality_metrics, compute_layer_assignments

centrality_df = compute_centrality_metrics(bg).reset_index()
layer_df = compute_layer_assignments(bg)

try:
    critical_path_df = ds.load_intermediate("critical_path")
    critical_path_targets = set(
        critical_path_df[critical_path_df["on_critical_path"]]["cmake_target"]
    )
except FileNotFoundError:
    critical_path_targets = set()

dep_path = export_dependency_graph(
    bg=bg,
    centrality=centrality_df,
    layers=layer_df,
    communities=best_communities_df,
    timing=tm,
    team_assignments=team_df,
    critical_path_targets=critical_path_targets,
    output_path=INTERMEDIATE_DIR / "gephi" / "dependency_graph.gexf",
)
print(f"\nRe-exported dependency graph with final communities: {dep_path}")